# Finetune ds distilled model

In [ ]:
import os
import json
import numpy as np
import pandas as pd

from unsloth import FastLanguageModel
from datasets import load_dataset


model_dir = '/root/autodl-tmp/models/DeepSeek-R1-Distill-Qwen-1.5B'
experiment = 'ds_r1_law'

new_model_local_dir = f'{experiment}_base'
print(f'new model local dir: {new_model_local_dir}')

new_merged_model_local_dir = f'{experiment}_merged'
print(f'new merged model local dir: {new_merged_model_local_dir}')

eval_result_dir = f"{experiment}_eval_result"
print(f'eval result save dir: {eval_result_dir}')


# data
# local_sft_data_path = '/root/autodl-tmp/dataset/medical-o1-reasoning-SFT/medical_o1_sft.json'
local_sft_data_path = '/root/autodl-tmp/dataset/finetune_processed_train.json'
eval_data_path = '/root/autodl-tmp/dataset/finetune_processed_eval.json'


# hyper parameter
max_seq_length = 2048 
dtype = None 
load_in_4bit = True
load_in_8bit, full_finetuning = False, False
learning_rate = 2e-4
num_train_epochs = 1
max_steps = -1

lora_rank = 16
lora_alpha = 16


# eval
text2vec_model_path = '/root/autodl-tmp/models/text2vec-base-chinese'
eval_sample_num = 5000

# Load model

In [ ]:
%%time

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_dir,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit, 
    load_in_8bit = load_in_8bit,
    full_finetuning = full_finetuning,
)


In [ ]:
prompt_style = """下面是一个法律咨询问题，请提供一个回复来解决咨询问题。
### 指令：
你是一个法律咨询专家，请回答以下问题，不需要提供思考过程。

### 问题：
{}

### 回复:
{}"""

train_prompt_style = """下面是一个法律咨询问题，请提供一个回复来解决咨询问题。
不需要提供思考过程
### 指令：
你是一个法律咨询专家，请回答以下问题，不需要提供思考过程。

### 问题：
{}

### 回复:
{}"""

In [ ]:
%%time

question = """
农村宅基地可以继承吗，需要办理什么手续才可以建房
"""

FastLanguageModel.for_inference(model) 
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=2000,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)
print(response[0].split("### 回复:")[1])

In [ ]:
EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    inputs = examples["question"]
    outputs = examples["anwser"]
    texts = []
    for inputs, outputs in zip(inputs, outputs):
        text = train_prompt_style.format(inputs, outputs) + EOS_TOKEN
        texts.append(text)
    return {
        "text": texts,
    }

In [ ]:
%%time


dataset = load_dataset("json", data_files=local_sft_data_path, split="train[:10]")

processed_dataset = dataset.map(formatting_prompts_func, batched = True,)
processed_dataset = processed_dataset
print('dataset example')
print(processed_dataset['text'][0])


# Finetune training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported


In [ ]:
%%time

peft_model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=lora_alpha,
    lora_dropout=0,  
    bias="none",
    use_gradient_checkpointing="unsloth",  # True or "unsloth" for very long context
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:

trainer = SFTTrainer(
    model=peft_model,
    tokenizer=tokenizer,
    train_dataset=processed_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        # Use num_train_epochs = 1, warmup_ratio for full training runs!
        warmup_steps=5,
        max_steps=max_steps,
        num_train_epochs=num_train_epochs,
        learning_rate=learning_rate,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

In [ ]:
%%time
trainer_stats = trainer.train()


wandb: Run data is saved locally in /root/autodl-tmp/ds_finetune/wandb/run-20250402_153414-bduruxlh
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run outputs


wandb: ⭐️ View project at https://wandb.ai/xxxgqcoder-individual-researcher/huggingface


wandb: 🚀 View run at https://wandb.ai/xxxgqcoder-individual-researcher/huggingface/runs/bduruxlh


Step,Training Loss
10,3.763000
20,2.191400
30,1.490900
40,1.067700
50,0.698700
60,0.448800


CPU times: user 39.6 s, sys: 287 ms, total: 39.9 s
Wall time: 42.4 s


In [ ]:
%%time

question = """
农村宅基地可以继承吗，需要办理什么手续才可以建房
"""

FastLanguageModel.for_inference(model) 
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=2000,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)
print(response[0].split("### 回复:")[1])

# Save model

In [ ]:
%%time

peft_model.save_pretrained(new_model_local_dir) 
tokenizer.save_pretrained(new_model_local_dir)

peft_model.save_pretrained_merged(new_merged_model_local_dir, tokenizer, save_method = "merged_16bit",)

print(f'finetuned model saved to {new_model_local_dir}, merged model saved to {new_merged_model_local_dir}')

Unsloth: Saving tokenizer... Done.


Done.


finetuned model saved to DeepSeek-R1-law
CPU times: user 8.97 s, sys: 4.48 s, total: 13.5 s
Wall time: 13.4 s


# Eval model

In [ ]:
from sentence_transformers import SentenceTransformer



In [ ]:
def _cos_sim(a, b):
    from numpy import dot
    from numpy.linalg import norm
    divider = norm(a) * norm(b)
    if abs(divider) < 1e-6:
        return 0
    return dot(a, b) / divider


def answer_sim(a, b, text2vec_model):
    if len(a) == 0 and len(b) == 0:
        return -1
    if len(a) == 0 or len(b) == 0:
        return -1

    embeddings = text2vec_model.encode([a, b])
    cos_sim = _cos_sim(embeddings[0], embeddings[1])
    return cos_sim

In [ ]:
%%time
trained_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = new_merged_model_local_dir,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit, 
    load_in_8bit = load_in_8bit,
    full_finetuning = full_finetuning,
)

FastLanguageModel.for_inference(trained_model) 
print(f'loaded trained model from {new_merged_model_local_dir}')


In [ ]:
text2vec_model = SentenceTransformer(text2vec_model_path)
print(f'loaded text2vec model from {text2vec_model_path}')


eval_result = pd.DataFrame(columns=['question', 'answer', 'pred', 'cos_sim'])

In [ ]:
%%time

with open(eval_data_path, ) as f:
    eval_data = json.load(f)

print(f'loaded eval data from {eval_data_path}, total {len(eval_data)} records')


In [ ]:
%%time

for i, data in enumerate(eval_data):
    if i >= eval_sample_num:
        break
    print(f'processing {i}')
    
    question = data['question']
    answer = data['answer']

    inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_seq_length,
        use_cache=True,
    )
    response = tokenizer.batch_decode(outputs)
    pred = response[0].split("### 回复:")[1]
    
    cos_sim = answer_sim(answer, pred, text2vec_model)

    row = {
        'question': question,
        'answer': answer,
        'pred': pred,
        'cos_sim': cos_sim,
    }
    eval_result.loc[len(eval_result)] = row


In [ ]:
print('average cos sim on eval data', eval_result['cos_sim'].mean())

os.makedirs(eval_result_dir, exist_ok=True)
eval_result.to_csv(os.path.join(eval_result_dir, 'eval_result.csv'), index=False)
print(f'eval result saved to {eval_result_dir}')